# Smart Tile Extraction — PANDA

Extracts the most tissue-dense tiles from each whole-slide image (WSI), replacing
naive grid sampling with tissue-aware selection. Tiles are ranked by tissue coverage
(HSV saturation + Otsu thresholding) and split at the slide level to avoid data leakage
between train and validation sets.

The extraction loop is resumable: progress is tracked on disk (`manifest.csv`,
`skipped_slides.csv`), so the notebook can be safely restarted at any point without
losing completed work or producing duplicate tiles.

In [1]:
import os
import cv2
import csv
import shutil
import tifffile
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from sklearn.model_selection import train_test_split

## Configuration

In [2]:
# Paths
DATA_DIR = "../data"
IMG_DIR = f"{DATA_DIR}/train_images"
CSV_PATH = f"{DATA_DIR}/train.csv"
TILES_DIR = f"{DATA_DIR}/tiles"
MANIFEST_PATH = f"{DATA_DIR}/manifest.csv"
SKIP_LOG_PATH = f"{DATA_DIR}/skipped_slides.csv"
FAILED_LOG_PATH = f"{DATA_DIR}/failed_slides.log"

# Split Configs
SEED = 42
VAL_SPLIT = 0.2

# Pipeline Configs
TIFF_LEVEL = 0
TILE_SIZE = 256
SAT_THRESHOLD = None
N_TILES = 36

## Output Directories

Created once, non-destructively. Existing tiles are never deleted, so a resumed
run's manifest always stays consistent with what is actually on disk.

In [3]:
for split in ["train", "val"]:
    for grade in range(6):
        Path(f"{TILES_DIR}/{split}/{grade}").mkdir(parents=True, exist_ok=True)

## Load Metadata and Train/Validation Split

Split is performed on slide IDs (`image_id`) before any tiling, and stratified by
ISUP grade so each split has proportional class representation.

In [4]:
df = pd.read_csv(CSV_PATH)

In [5]:
train_ids, val_ids = train_test_split(
    df["image_id"],
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=df["isup_grade"],
    shuffle=True,
)

In [6]:
train_ids = set(train_ids)
val_ids = set(val_ids)

In [7]:
# image_id -> {isup_grade, data_provider}
slide_info = df.set_index("image_id")[["isup_grade", "data_provider"]].to_dict("index")

## Tissue Detection

In [8]:
def get_tissue_mask(rgb_image: np.ndarray, sat_threshold: int | None = None) -> np.ndarray:
    """
    Segment tissue from background using the saturation channel in HSV space.
    H&E-stained tissue is highly saturated; background glass is near zero.
    If sat_threshold is None, Otsu's method selects the threshold automatically.
    """
    if rgb_image.dtype != np.uint8:
        rgb_image = (rgb_image / rgb_image.max() * 255).astype(np.uint8)

    hsv        = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2HSV)
    saturation = hsv[:, :, 1]

    if sat_threshold is None:
        _, mask = cv2.threshold(saturation, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        _, mask = cv2.threshold(saturation, sat_threshold, 255, cv2.THRESH_BINARY)

    return mask.astype(bool)

## Tile Extraction

In [9]:
def extract_tiles(
    image_id: str,
    n_tiles: int = N_TILES,
    level: int = TIFF_LEVEL,
    tile_size: int = TILE_SIZE,
    sat_threshold: int | None = SAT_THRESHOLD,
) -> list[tuple]:
    """
    Extract the top-N most tissue-rich tiles from a single WSI.

    The tissue threshold is computed once per slide from the lowest-resolution
    pyramid level (a few MB) rather than the full-resolution image, so Otsu's
    method never has to run on a multi-GB array. This is an approximation of
    the full-resolution threshold (downsampling smooths the histogram slightly),
    but the difference is negligible for tissue/background separation.

    During the grid scan, only (coverage, x, y) is kept per candidate tile —
    pixel data is deliberately not copied at this stage. Copying every
    tissue-positive tile up front would scale with tissue density (a dense
    slide can have thousands of valid tiles), silently adding hundreds of MB
    to a GB of overhead on top of the base image. Pixel data is only copied
    for the final n_tiles selections, once the scan is complete.

    Returns (coverage, tile, x, y) tuples sorted by coverage descending.
    coverage is the fraction of tile pixels classified as tissue (0.0-1.0);
    x, y is the tile's top-left corner in the given pyramid level's coordinates.
    """
    path = os.path.join(IMG_DIR, f"{image_id}.tiff")

    with tifffile.TiffFile(path) as tif:
        img = tif.series[0].levels[level].asarray()

        # Derive the global saturation threshold from the thumbnail level so
        # Otsu's method operates on a few-MB array instead of the full slide.
        if sat_threshold is None:
            thumb = tif.series[0].levels[-1].asarray()
            if thumb.ndim == 3 and thumb.shape[2] == 4:
                thumb = thumb[:, :, :3]
            if thumb.dtype != np.uint8:
                thumb = (thumb / thumb.max() * 255).astype(np.uint8)

            thumb_hsv = cv2.cvtColor(thumb, cv2.COLOR_RGB2HSV)
            global_thresh, _ = cv2.threshold(
                thumb_hsv[:, :, 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )
        else:
            global_thresh = sat_threshold

    # Drop alpha channel if present
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]

    H, W = img.shape[:2]

    # Scan the grid, scoring every tile's tissue coverage against the global
    # threshold. Only the score and coordinates are kept here — no pixel data —
    # so memory use stays flat regardless of how many tiles pass the check.
    candidates = []

    for y in range(0, H - tile_size + 1, tile_size):
        for x in range(0, W - tile_size + 1, tile_size):
            tile = img[y : y + tile_size, x : x + tile_size]

            mask = get_tissue_mask(tile, sat_threshold=global_thresh)
            coverage = mask.mean()

            if coverage > 0:
                candidates.append((coverage, x, y))

    candidates.sort(key=lambda t: t[0], reverse=True)
    top_coords = candidates[:n_tiles]

    # Only now do we materialize pixel data — and only for the tiles we're
    # actually keeping, capping this step at n_tiles copies regardless of
    # slide density.
    final_tiles = [
        (coverage, img[y : y + tile_size, x : x + tile_size].copy(), x, y)
        for coverage, x, y in top_coords
    ]

    return final_tiles

## Pre-Run Checks

Smoke test on a handful of slides, plus a disk space check, before committing to
the full multi-hour run.

In [10]:
test_ids = list(slide_info.keys())[:5]
for image_id in tqdm(test_ids, desc="Smoke test"):
    try:
        tiles = extract_tiles(image_id)
        avg_tissue = np.mean([t[0] for t in tiles]) if tiles else 0.0
        print(f"{image_id}: {len(tiles)} tiles, avg tissue={avg_tissue:.2%}")
    except Exception as e:
        print(f"FAILED: {image_id} — {e}")

Smoke test:  20%|██        | 1/5 [00:01<00:06,  1.70s/it]

0005f7aaab2800f6170c399693a96917: 36 tiles, avg tissue=98.40%


Smoke test:  40%|████      | 2/5 [00:02<00:02,  1.03it/s]

000920ad0b612851f8e01bcc880d9b3d: 36 tiles, avg tissue=96.19%


Smoke test:  60%|██████    | 3/5 [00:02<00:01,  1.50it/s]

0018ae58b01bdadc8e347995b69f99aa: 36 tiles, avg tissue=98.99%


Smoke test:  80%|████████  | 4/5 [00:03<00:00,  1.05it/s]

001c62abd11fa4b57bf7a6c603a11bb9: 36 tiles, avg tissue=97.70%


Smoke test: 100%|██████████| 5/5 [00:05<00:00,  1.19s/it]

001d865e65ef5d2579c190a0e0350d8f: 36 tiles, avg tissue=99.30%


In [11]:
total, used, free = shutil.disk_usage(DATA_DIR)
print(f"Free space: {free / 1e9:.1f} GB")

Free space: 510.6 GB


## Resumable Extraction

Progress is tracked on disk rather than in memory, so a kernel restart does not
repeat completed work:

- `manifest.csv` — one row per saved tile, appended after each slide completes.
- `skipped_slides.csv` — slides with zero usable tiles; excluded permanently.
- `failed_slides.log` — slides that raised an exception; **not** excluded, since
  the failure may be transient (e.g. a disk or memory issue) and should retry.

In [12]:
manifest_fields = [
    "tile_path", "image_id", "isup_grade", "split",
    "tile_index", "data_provider", "x", "y", "tissue_pct",
]
skip_fields = ["image_id", "reason"]

if not Path(MANIFEST_PATH).exists():
    with open(MANIFEST_PATH, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=manifest_fields).writeheader()

if not Path(SKIP_LOG_PATH).exists():
    with open(SKIP_LOG_PATH, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=skip_fields).writeheader()

# Read progress from disk, not memory, so this is correct even after a restart.
try:
    existing_manifest = pd.read_csv(MANIFEST_PATH)
except pd.errors.EmptyDataError:
    existing_manifest = pd.DataFrame(columns=manifest_fields)
processed_ids = set(existing_manifest["image_id"].unique()) if not existing_manifest.empty else set()

try:
    existing_skips = pd.read_csv(SKIP_LOG_PATH)
except pd.errors.EmptyDataError:
    existing_skips = pd.DataFrame(columns=skip_fields)
skipped_ids = set(existing_skips["image_id"].unique()) if not existing_skips.empty else set()

already_done = processed_ids | skipped_ids
if already_done:
    print(f"Resuming: {len(processed_ids)} slides already extracted, {len(skipped_ids)} previously skipped.")

Resuming: 3313 slides already extracted, 1 previously skipped.


In [13]:
for image_id, info in tqdm(slide_info.items(), total=len(slide_info), desc="Extracting"):
    if image_id in already_done:
        continue

    grade = info["isup_grade"]
    provider = info["data_provider"]
    split = "train" if image_id in train_ids else "val"

    try:
        tiles = extract_tiles(image_id)
    except Exception as e:
        print(f"FAILED: {image_id} — {e}")
        with open(FAILED_LOG_PATH, "a") as f:
            f.write(f"{image_id}\t{e}\n")
        continue

    if len(tiles) == 0:
        print(f"WARNING: 0 tiles extracted for {image_id} ({provider}, grade={grade})")
        with open(SKIP_LOG_PATH, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=skip_fields).writerow(
                {"image_id": image_id, "reason": "zero_tiles"}
            )
        continue

    slide_rows = []
    for i, (coverage, tile, x, y) in enumerate(tiles):
        filename = f"{image_id}_t{i:02d}.png"
        save_path = f"{TILES_DIR}/{split}/{grade}/{filename}"

        ok = cv2.imwrite(save_path, cv2.cvtColor(tile, cv2.COLOR_RGB2BGR))
        if not ok:
            print(f"WRITE ERROR: {save_path}")
            continue

        slide_rows.append({
            "tile_path": save_path,
            "image_id": image_id,
            "isup_grade": grade,
            "split": split,
            "tile_index": i,
            "data_provider": provider,
            "x": x,
            "y": y,
            "tissue_pct": round(coverage, 4),
        })

    if slide_rows:
        with open(MANIFEST_PATH, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=manifest_fields).writerows(slide_rows)

print("\nExtraction loop finished.")

Extracting: 100%|██████████| 10616/10616 [2:02:37<00:00,  1.44it/s] 


Extraction loop finished.


## Summary

In [14]:
manifest_df = pd.read_csv(MANIFEST_PATH)
print(
    f"Done. {len(manifest_df):,} tiles saved across {manifest_df['image_id'].nunique():,} slides."
)
print(manifest_df.groupby(["split", "isup_grade"]).size().unstack(fill_value=0))

Done. 382,117 tiles saved across 10,615 slides.
isup_grade      0      1      2      3      4      5
split                                               
train       83263  76788  38664  35784  35956  35241
val         20808  19188   9684   8928   8993   8820
